# Fake News Classification Model using LSTM

This notebook demonstrates a complete end-to-end pipeline for classifying news as fake or real using the **WELFake Dataset**. Below is an overview of the dataset preprocessing, architecture design, and final evaluation metrics.

---

## 1. Dataset & Preprocessing
- **Source**: `saurabhshahane/fake-news-classification` via KaggleHub.
- **Feature Engineering**: Concatenated `title` and `text` columns into a single unified `news` column.
- **Text Cleaning**:
  - Lowercasing and removal of special characters using regular expressions.
  - Tokenization and lemmatization utilizing NLTK's `WordNetLemmatizer`.
  - Stopword removal using standard NLTK English stopwords.
- **Vocabulary Size**: 230,533 unique words.
- **Sequence Representation**: Word token indexes with padding set to a max length of `100` words.

---

## 2. Model Architecture
We constructed a Sequential Deep Learning model in TensorFlow/Keras using the following architecture:

1. **Embedding Layer**: Projects the vocabulary index into a dense vector space of `50` dimensions.
2. **LSTM Layer**: Recurrent layer with `100` memory units to capture sequential temporal relationships.
3. **Dense Layer (Output)**: Single neuron output with a `sigmoid` activation function for binary classification.
4. **Optimization**: Compiled using the `adam` optimizer and `binary_crossentropy` loss.
5. **Regularization**: Integrated `EarlyStopping` monitoring validation loss to prevent overfitting.

---

## 3. Evaluation & Performance Report
The model achieved **93.65% Accuracy** on the validation split (33% test size) with early stopping completing the training at epoch 7.

### Confusion Matrix
```python
[[10920,   726]   # [True Negatives, False Positives]
 [  773, 11189]]  # [False Negatives, True Positives]
```

### Classification Report
```text
              precision    recall  f1-score   support

           0       0.94      0.93      0.94     11693
           1       0.94      0.94      0.94     11915

    accuracy                           0.94     23608
   macro avg       0.94      0.94      0.94     23608
weighted avg       0.94      0.94      0.94     23608
```

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "WELFake_Dataset.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "saurabhshahane/fake-news-classification",
  file_path,
)

df.head(5)

/tmp/ipykernel_389/2277613062.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'fake-news-classification' dataset.


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [2]:
import pandas as pd
import numpy as np

In [3]:
df.drop(columns={"Unnamed: 0"},inplace=True)
df.head(5)

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [4]:
df = df.dropna()

In [5]:
df.shape

(71537, 3)

In [6]:
df.isnull().sum()

,0
title,0
text,0
label,0


In [7]:
df.head(5)

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1


In [8]:
df['news'] = df['title'] + df['text']
df.drop(columns=['title','text'],inplace=True)
df.head(5)

,label,news
0,1,LAW ENFORCEMENT ON HIGH ALERT Following Threat...
2,1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3,0,"Bobby Jindal, raised Hindu, uses story of Chri..."
4,1,SATAN 2: Russia unvelis an image of its terrif...
5,1,About Time! Christian Group Sues Amazon and SP...


In [9]:
X = df['news']
y = df['label']

In [10]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

corpus = []
lemmatizer = WordNetLemmatizer()

stop_words  = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [26]:
len(stop_words)

198

In [11]:
import re

for text in X:
  review = re.sub('[^a-zA-Z0-9]',' ', text)
  review = review.lower()
  review = review.split()
  review = [lemmatizer.lemmatize(word) for word in review if not word in stop_words]
  review = ' '.join(review)
  corpus.append(review)

In [12]:
print(len(corpus),y.shape[0])

71537 71537


In [13]:
a = set()
for sent in corpus:
  for word in sent.split():
    a.add(word)

len(a)

230533

In [14]:
vocab_size = len(a)

In [15]:
import tensorflow as tf

In [16]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM , Dense

In [17]:
one_hot_rep = [one_hot(words,vocab_size) for words in corpus]

In [18]:
sent_len = 100

In [19]:
embeded_docs = pad_sequences(one_hot_rep,padding='post',maxlen=sent_len,truncating='post')

In [20]:
embeded_docs[0]

array([137703, 140583, 158411,  63793,  62924, 181683,  78183,   4890,
       195590, 147957, 214103,  65078,  17749,  28425, 152363,  44079,
       156901,  77475, 186110,  65078, 104093, 214103, 119503,  16278,
       157901, 133848,   4890, 176348,  78183,   4082,  99203,  76754,
       115037, 225402,  98283, 229846, 194840,  87602,   4890, 176348,
        78183, 182465,   4595, 194601, 164183, 176348, 131985, 221875,
       131052, 116996,  67026,  16278, 185236,  76754, 146880, 115037,
       158669,  29437,  16278, 185236, 131052, 179946,  91680,  76754,
       115037, 222580,  65078, 119871, 140191,  58866, 195590,  11835,
        56408, 115037, 115708, 149855, 224482,  65078, 215497, 227275,
        70382, 138945,   4890, 222580, 140191,  76754, 224482,  99046,
       162670,  76754, 115037, 146899,  15790,  16608,  56408, 157265,
        51869,  94106, 115037, 178248], dtype=int32)

In [21]:
features = 50
model = Sequential()
model.add(Embedding(vocab_size,features,input_length=sent_len))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [22]:
X_final = np.array(embeded_docs)

In [23]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X_final,y,test_size=0.33,random_state=42)

In [24]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [25]:
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64,callbacks=[early_stopping])

Epoch 1/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.7478 - loss: 0.5360 - val_accuracy: 0.7114 - val_loss: 0.5927
Epoch 2/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.7671 - loss: 0.5009 - val_accuracy: 0.7925 - val_loss: 0.4151
Epoch 3/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.8118 - loss: 0.4195 - val_accuracy: 0.8544 - val_loss: 0.3836
Epoch 4/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.8903 - loss: 0.2902 - val_accuracy: 0.8868 - val_loss: 0.3083
Epoch 5/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.9429 - loss: 0.1662 - val_accuracy: 0.9165 - val_loss: 0.2313
Epoch 6/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - accuracy: 0.9720 - loss: 0.0862 - val_accuracy: 0.9330 - val_loss: 0.2158
Epoch 7/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.9879 - loss: 0.0413 - val_accuracy: 0.9365 - val_loss: 0.2047
Epoch 8/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.9944 - loss: 0.0208 - 

In [27]:
y_pred = model.predict(X_test)

738/738 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step


In [28]:
y_pred = np.where(y_pred > 0.5,1,0)

In [29]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_pred,y_test))

[[10920   726]
 [  773 11189]]


In [31]:
from sklearn.metrics import accuracy_score,classification_report
accuracy_score(y_pred,y_test)

0.9365045747204338

In [33]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.94      0.93      0.94     11693
           1       0.94      0.94      0.94     11915

    accuracy                           0.94     23608
   macro avg       0.94      0.94      0.94     23608
weighted avg       0.94      0.94      0.94     23608

